In [12]:
import json
import os
import re
import html
from pathlib import Path
from typing import List, Tuple, Optional, Dict
from collections import defaultdict

try:
    from doc_store import DocClient
    from doc_store.interface import TaggingInput, TaskInput
    HAS_DOC_STORE = True
except ImportError:
    HAS_DOC_STORE = False
    DocClient = None


def strip_html_tags(text: str) -> str:
    """去除HTML标签并还原HTML实体"""
    if not text:
        return text
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = text.strip()
    return text


def levenshtein_distance(s1: str, s2: str) -> int:
    """计算两个字符串之间的编辑距离（Levenshtein Distance）"""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]


def normalized_edit_distance(s1: str, s2: str) -> float:
    """计算归一化的编辑距离得分（0-1之间，1表示完全匹配）"""
    if not s1 and not s2:
        return 1.0
    
    edit_dist = levenshtein_distance(s1, s2)
    max_len = max(len(s1), len(s2))
    
    return 1.0 - (edit_dist / max_len)


def get_gt_from_store(store: DocClient, block_id: str, version: str) -> Optional[str]:
    """
    从 doc_store 获取 GT 内容
    
    Args:
        store: DocClient 实例
        block_id: block ID
        version: 版本号 (label_task_id)
    
    Returns:
        GT 文本内容，获取失败返回 None
    """
    try:
        result = store.try_get_content_by_block_id_and_version(block_id, version)
        if result and result.content:
            return result.content
    except Exception as e:
        print(f"警告: 从 doc_store 获取 GT 失败 (block_id={block_id}, version={version}): {e}")
    return None


def extract_texts_from_jsonl(
    jsonl_path: str, 
    store: DocClient = None,
    gt_version: str = None
) -> List[Dict]:
    """
    从jsonl文件中提取pred文本，从doc_store获取gt文本
    
    Args:
        jsonl_path: jsonl 文件路径
        store: DocClient 实例，用于获取 GT
        gt_version: GT 版本号，若为 None 则使用 jsonl 中的 label_task_id
    
    Returns:
        List of dicts with keys: data_id, pred_text, gt_text, status, block_id, creator
    """
    results = []
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            
            try:
                data = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"警告: 第 {line_num} 行 JSON 解析失败: {e}")
                continue
            
            data_id = data.get('data_id', f'unknown_{line_num}')
            status = data.get('status', 'unknown')
            creator = data.get('creator', 'unknown')
            
            # 提取 custom 信息
            custom = data.get('custom', {}) or {}
            block_id = custom.get('block_id', None)
            label_task_id = custom.get('label_task_id', None)
            
            # 提取 pred text（需要清理HTML标签）
            pred_text = None
            try:
                evaluation = data.get('evaluation', {}) or {}
                conv_eval = evaluation.get('conversation_evaluation', {}) or {}
                eval_contents = conv_eval.get('contents', []) or []
                if eval_contents and len(eval_contents) > 0:
                    pred_text = eval_contents[0].get('content', None)
                    if pred_text:
                        pred_text = strip_html_tags(pred_text)
            except (TypeError, KeyError, IndexError, AttributeError):
                pass
            
            # 从 doc_store 获取 gt text
            gt_text = None
            if store and block_id:
                version = gt_version or label_task_id
                if version:
                    gt_text = get_gt_from_store(store, block_id, version)
            
            # 如果 doc_store 获取失败，回退到 jsonl 中的 reference_evaluation
            if gt_text is None:
                try:
                    ref_evaluation = data.get('reference_evaluation', {}) or {}
                    ref_conv_eval = ref_evaluation.get('conversation_evaluation', {}) or {}
                    ref_contents = ref_conv_eval.get('contents', []) or []
                    if ref_contents and len(ref_contents) > 0:
                        gt_text = ref_contents[0].get('content', None)
                except (TypeError, KeyError, IndexError, AttributeError):
                    pass
            
            results.append({
                'data_id': data_id,
                'pred_text': pred_text,
                'gt_text': gt_text,
                'status': status,
                'block_id': block_id,
                'creator': creator
            })
    
    return results


def evaluate_file(
    jsonl_path: str, 
    store: DocClient = None,
    gt_version: str = None,
    verbose: bool = True
) -> dict:
    """
    评估单个jsonl文件，按 creator 分组统计
    
    Args:
        jsonl_path: jsonl 文件路径
        store: DocClient 实例，用于获取 GT
        gt_version: GT 版本号
        verbose: 是否输出详细信息
    
    Returns:
        dict: 包含评估结果的字典
    """
    results = extract_texts_from_jsonl(jsonl_path, store=store, gt_version=gt_version)
    
    file_name = os.path.basename(jsonl_path)
    
    # 按 creator 分组
    creator_data = defaultdict(list)
    skipped = 0
    
    for item in results:
        data_id = item['data_id']
        pred_text = item['pred_text']
        gt_text = item['gt_text']
        status = item['status']
        creator = item['creator']
        
        # 只评估已完成的标注
        if status != 'completed':
            skipped += 1
            continue
        
        # 检查是否有有效的pred和gt
        if pred_text is None or gt_text is None:
            skipped += 1
            continue
        
        # 计算得分
        score = normalized_edit_distance(pred_text, gt_text)
        
        creator_data[creator].append({
            'data_id': data_id,
            'score': score,
            'pred_text': pred_text,
            'gt_text': gt_text
        })
    
    # 输出结果
    if verbose:
        print(f"\n{'='*80}")
        print(f"文件: {file_name}")
        print(f"{'='*80}")
    
    creator_stats = {}
    all_scores = []
    
    for creator, items in sorted(creator_data.items()):
        scores = [item['score'] for item in items]
        all_scores.extend(scores)
        
        avg_score = sum(scores) / len(scores) if scores else 0.0
        min_score = min(scores) if scores else 0.0
        max_score = max(scores) if scores else 0.0
        
        creator_stats[creator] = {
            'count': len(items),
            'avg_score': avg_score,
            'min_score': min_score,
            'max_score': max_score,
            'scores': scores
        }
        
        if verbose:
            print(f"\n{'-'*80}")
            print(f"Creator: {creator}")
            print(f"{'-'*80}")
            print(f"  题目数: {len(items)} | 平均分: {avg_score:.4f} | 最低分: {min_score:.4f} | 最高分: {max_score:.4f}")
            print(f"  题目明细:")
            
            for idx, item in enumerate(items, 1):
                score_str = f"{item['score']:.4f}"
                data_id_short = item['data_id'][:36]
                if item['score'] < 1.0:
                    # 显示差异信息，处理可能的编码问题
                    def safe_preview(text, max_len=40):
                        text = text.replace('\n', '\\n')
                        if len(text) > max_len:
                            text = text[:max_len] + '...'
                        return text.encode('gbk', errors='replace').decode('gbk')
                    
                    gt_preview = safe_preview(item['gt_text'])
                    pred_preview = safe_preview(item['pred_text'])
                    print(f"    {idx:3}. [{score_str}] {data_id_short}")
                    print(f"         GT:   {gt_preview}")
                    print(f"         Pred: {pred_preview}")
                else:
                    print(f"    {idx:3}. [{score_str}] {data_id_short}")
            
            print(f"{'-'*80}")
    
    # 总体统计
    total_completed = len(all_scores)
    overall_avg = sum(all_scores) / len(all_scores) if all_scores else 0.0
    overall_min = min(all_scores) if all_scores else 0.0
    overall_max = max(all_scores) if all_scores else 0.0
    
    if verbose:
        print(f"\n{'='*80}")
        print(f"文件总计: {file_name}")
        print(f"{'='*80}")
        print(f"  总样本数: {len(results)}")
        print(f"  已评估: {total_completed}")
        print(f"  跳过: {skipped}")
        print(f"  Creator 数: {len(creator_data)}")
        print(f"  总平均得分: {overall_avg:.4f}")
        print(f"  总最低得分: {overall_min:.4f}")
        print(f"  总最高得分: {overall_max:.4f}")
    
    return {
        'file': file_name,
        'total': len(results),
        'completed': total_completed,
        'skipped': skipped,
        'avg_score': overall_avg,
        'min_score': overall_min,
        'max_score': overall_max,
        'scores': all_scores,
        'creator_stats': creator_stats,
        'creator_data': dict(creator_data)
    }


def evaluate_folder(
    folder_path: str, 
    store: DocClient = None,
    gt_version: str = None,
    verbose: bool = True
) -> List[dict]:
    """
    评估文件夹下所有jsonl文件
    
    Args:
        folder_path: 文件夹路径
        store: DocClient 实例，用于获取 GT
        gt_version: GT 版本号
        verbose: 是否输出详细信息
    """
    folder = Path(folder_path)
    jsonl_files = list(folder.glob('*.jsonl'))
    
    if not jsonl_files:
        print(f"在 {folder_path} 中未找到 jsonl 文件")
        return []
    
    print(f"找到 {len(jsonl_files)} 个 jsonl 文件")
    
    all_results = []
    all_scores = []
    
    for jsonl_file in jsonl_files:
        result = evaluate_file(str(jsonl_file), store=store, gt_version=gt_version, verbose=verbose)
        all_results.append(result)
        all_scores.extend(result['scores'])
    
    # 总体统计
    if all_scores:
        print(f"\n{'='*60}")
        print("总体统计")
        print(f"{'='*60}")
        print(f"总评估样本数: {len(all_scores)}")
        print(f"总平均得分: {sum(all_scores)/len(all_scores):.4f}")
        print(f"总最低得分: {min(all_scores):.4f}")
        print(f"总最高得分: {max(all_scores):.4f}")
    
    return all_results




In [ ]:
if __name__ == '__main__':
    # ============ 配置参数 ============
    INPUT_PATH = 'anno_result'                          # 输入文件或文件夹路径
    VERBOSE = True                                      # 是否输出详细信息
    DOC_STORE_URL = "http://docs.bigdata.shlab.tech:8080"  # doc_store 服务地址
    GT_VERSION = None                                   # GT 版本号，None 则使用 jsonl 中的 label_task_id
    # =================================
    
    # 初始化 doc_store 客户端
    store = DocClient(DOC_STORE_URL)
    
    input_path = "/share/quyuan/notebooks/quyuan/SingleDocBench/text/anno_result"
 
  
    evaluate_folder(str(input_path), store=store, gt_version=GT_VERSION, verbose=VERBOSE)
    